# Statistical Hypothesis Testing

**Project**: WGU D610 Capstone - Serverless FHIR Pipeline Performance Study  
**Author**: Robert Miller  
**Date**: November 2025  
**Purpose**: Execute formal statistical tests to evaluate pipeline performance hypotheses

---

## Overview

This notebook executes the primary statistical analyses:

1. **Paired t-test**: Compare automated vs manual latency (H₁)
2. **McNemar's test**: Compare automated vs manual validation accuracy (H₂)
3. **Effect Size**: Calculate Cohen's d for practical significance
4. **Sensitivity Analysis**: Test robustness across baseline assumptions
5. **Subgroup Analysis**: Examine performance across data types
6. **Assumption Validation**: Check statistical prerequisites

**Research Questions**:
- **H₁**: Does the automated pipeline process records faster than manual baseline?
- **H₂**: Does the automated pipeline validate data more accurately than manual baseline?

## 1. Setup and Configuration

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.contingency_tables import mcnemar
from pathlib import Path
import warnings

# Suppress warnings
warnings.filterwarnings('ignore')

# Set random seed
np.random.seed(12345)

# Configure plotting
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Create output directory
output_dir = Path('reports/figures/results')
output_dir.mkdir(parents=True, exist_ok=True)

# Significance level
ALPHA = 0.05

print("✓ Libraries imported and configuration complete")
print(f"✓ Output directory: {output_dir}")
print(f"✓ Significance level (α): {ALPHA}")

## 2. Load Data

In [ ]:
# Load processed data
data_path = Path('data/processed/combined_data.parquet')

if not data_path.exists():
    raise FileNotFoundError(
        f"Processed data not found at {data_path}. "
        "Please run Phase 3 data cleaning first."
    )

df = pd.read_parquet(data_path)

print(f"✓ Loaded {len(df):,} records")
print(f"✓ Data sources: {df['data_source'].nunique()}")

## 3. Hypothesis 1: Latency Improvement (Paired t-test)

**Research Question**: Does the automated pipeline process records faster than manual baseline?

**Hypotheses**:
- H₀: μ_diff = 0 (no difference between manual and automated latency)
- H₁: μ_diff > 0 (automated is faster than manual)

**Test**: One-tailed paired t-test  
**Baseline**: 20 minutes (1200 seconds) per record - conservative estimate from literature  
**Significance Level**: α = 0.05

### 3.1 Create Paired Comparison

In [ ]:
# Manual baseline from literature review (Phase 2)
# Arndt et al. (2017): 16.8-35 min/encounter
# Casey et al. (2019): 18-24 min/record
# Baseline: 20 min (conservative middle-range)

MANUAL_BASELINE_MIN = 20
MANUAL_BASELINE_SEC = MANUAL_BASELINE_MIN * 60

# Create paired data
df['manual_latency_est'] = MANUAL_BASELINE_SEC
df['latency_diff'] = df['manual_latency_est'] - df['latency_sec']  # Positive = improvement

print(f"Manual Baseline: {MANUAL_BASELINE_SEC} seconds ({MANUAL_BASELINE_MIN} minutes)")
print(f"\nAutomated Pipeline Statistics:")
print(f"  Mean: {df['latency_sec'].mean():.2f} seconds")
print(f"  Median (P50): {df['latency_sec'].median():.2f} seconds")
print(f"  P95: {df['latency_sec'].quantile(0.95):.2f} seconds")
print(f"\nDifference (Improvement):")
print(f"  Mean: {df['latency_diff'].mean():.2f} seconds")
print(f"  Std Dev: {df['latency_diff'].std():.2f} seconds")
print(f"  Min: {df['latency_diff'].min():.2f} seconds")
print(f"  Max: {df['latency_diff'].max():.2f} seconds")

### 3.2 Execute Paired t-test

In [ ]:
# Paired t-test (two-tailed, then convert to one-tailed)
t_stat, p_value_two_tail = stats.ttest_rel(df['manual_latency_est'], df['latency_sec'])

# One-tailed test (we expect automated < manual, so diff > 0)
p_value_one_tail = p_value_two_tail / 2 if t_stat > 0 else 1 - (p_value_two_tail / 2)

print("\n" + "="*70)
print("PAIRED T-TEST: LATENCY IMPROVEMENT (H₁)")
print("="*70)
print(f"Sample Size: {len(df):,}")
print(f"Degrees of Freedom: {len(df) - 1:,}")
print(f"\nt-statistic: {t_stat:.4f}")
print(f"p-value (two-tailed): {p_value_two_tail:.6f}")
print(f"p-value (one-tailed): {p_value_one_tail:.6f}")

# Decision
if p_value_one_tail < ALPHA:
    print(f"\n✓ REJECT H₀ (p < {ALPHA})")
    print("   Conclusion: Automated pipeline is SIGNIFICANTLY FASTER than manual baseline")
    print(f"   Mean improvement: {df['latency_diff'].mean():.2f} seconds ({df['latency_diff'].mean()/60:.2f} minutes)")
else:
    print(f"\n❌ FAIL TO REJECT H₀ (p >= {ALPHA})")
    print("   Conclusion: Insufficient evidence of speed improvement")

# Confidence interval for mean difference
ci = stats.t.interval(0.95, len(df)-1, 
                      loc=df['latency_diff'].mean(), 
                      scale=stats.sem(df['latency_diff']))

print(f"\n95% Confidence Interval for Mean Difference:")
print(f"  [{ci[0]:.2f}, {ci[1]:.2f}] seconds")
print(f"  [{ci[0]/60:.2f}, {ci[1]/60:.2f}] minutes")

### 3.3 Effect Size (Cohen's d)

In [ ]:
# Cohen's d for paired samples
mean_diff = df['latency_diff'].mean()
std_diff = df['latency_diff'].std()
cohens_d = mean_diff / std_diff

print("\n" + "="*70)
print("EFFECT SIZE ANALYSIS")
print("="*70)
print(f"Cohen's d: {cohens_d:.4f}")

# Interpretation (Cohen, 1988)
if abs(cohens_d) < 0.2:
    interpretation = "negligible"
    color = "⚠"
elif abs(cohens_d) < 0.5:
    interpretation = "small"
    color = "⚠"
elif abs(cohens_d) < 0.8:
    interpretation = "medium"
    color = "✓"
else:
    interpretation = "large"
    color = "✓"

print(f"\nInterpretation: {interpretation.upper()} effect")
print(f"\nBenchmarks (Cohen, 1988):")
print(f"  Small:    d = 0.2")
print(f"  Medium:   d = 0.5")
print(f"  Large:    d = 0.8")

if cohens_d >= 0.8:
    print(f"\n{color} Effect size meets success threshold (d ≥ 0.8)")
    print("   Result: LARGE practical significance")
elif cohens_d >= 0.5:
    print(f"\n{color} Effect size shows medium practical significance")
else:
    print(f"\n{color} Effect size below target (d = {cohens_d:.2f} < 0.8)")
    print("   Result: Limited practical significance")

# Speedup factor
speedup = df['manual_latency_est'].mean() / df['latency_sec'].mean()
print(f"\nSpeedup Factor: {speedup:.1f}x faster")
print(f"Time Savings: {(1 - 1/speedup) * 100:.1f}% reduction")

### 3.4 Visualization: Before-After Comparison

In [ ]:
# Create before-after comparison plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel A: Box plot comparison
data_for_plot = pd.DataFrame({
    'Manual\n(Baseline)': df['manual_latency_est'],
    'Automated\n(Measured)': df['latency_sec']
})

bp = axes[0].boxplot([data_for_plot['Manual\n(Baseline)'], data_for_plot['Automated\n(Measured)']], 
                      labels=['Manual\n(Baseline)', 'Automated\n(Measured)'],
                      patch_artist=True)

# Color boxes
bp['boxes'][0].set_facecolor('lightcoral')
bp['boxes'][1].set_facecolor('lightgreen')

axes[0].set_ylabel('Latency (seconds)', fontsize=12)
axes[0].set_title('Before-After Latency Comparison', fontsize=14, weight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

# Add statistics annotation
stats_text = f'p={p_value_one_tail:.6f}\nCohen\'s d={cohens_d:.2f}\nSpeedup: {speedup:.1f}x'
axes[0].text(0.5, 0.95, stats_text, 
             transform=axes[0].transAxes, ha='center', va='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8),
             fontsize=10)

# Panel B: Difference distribution
axes[1].hist(df['latency_diff'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[1].axvline(0, color='red', linestyle='--', linewidth=2, label='No Difference')
axes[1].axvline(df['latency_diff'].mean(), color='darkblue', linestyle='-', linewidth=2, 
                label=f'Mean Diff = {df["latency_diff"].mean():.1f}s')
axes[1].set_xlabel('Latency Difference (manual - automated, seconds)', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Distribution of Improvement', fontsize=14, weight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'paired_ttest_visualization.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Saved: paired_ttest_visualization.png")

## 4. Hypothesis 2: Validation Accuracy (McNemar's Test)

**Research Question**: Does the automated pipeline validate data more accurately than manual baseline?

**Hypotheses**:
- H₀: Proportion discordant pairs are equal (no difference in accuracy)
- H₁: Proportions differ (automated has different accuracy than manual)

**Test**: McNemar's test for paired nominal data  
**Baseline**: 85% validation pass rate - conservative estimate from Goldberg et al. (2008)  
**Significance Level**: α = 0.05

### 4.1 Create Simulated Manual Baseline

In [ ]:
# Manual baseline validation rate from literature (Goldberg et al., 2008)
MANUAL_VALIDATION_RATE = 0.85  # 85% pass rate

# Simulate manual validation results (for demonstration)
# In real study, this would be measured
np.random.seed(12345)
df['manual_validation_est'] = np.random.rand(len(df)) < MANUAL_VALIDATION_RATE

print(f"Manual Baseline Validation Rate: {MANUAL_VALIDATION_RATE * 100:.1f}%")
print(f"Automated Measured Validation Rate: {df['validation_pass_bool'].mean() * 100:.2f}%")
print(f"\nImprovement: {(df['validation_pass_bool'].mean() - MANUAL_VALIDATION_RATE) * 100:+.2f} percentage points")

### 4.2 Create Contingency Table

In [ ]:
# Create 2x2 contingency table
contingency = pd.crosstab(df['manual_validation_est'], df['validation_pass_bool'],
                         rownames=['Manual (Baseline)'], colnames=['Automated (Measured)'])

print("\n=== Contingency Table ===")
print(contingency)
print(f"\nTotal Records: {contingency.sum().sum()}")

# Concordant vs Discordant pairs
both_pass = contingency.loc[True, True] if True in contingency.index and True in contingency.columns else 0
both_fail = contingency.loc[False, False] if False in contingency.index and False in contingency.columns else 0
manual_pass_auto_fail = contingency.loc[True, False] if True in contingency.index and False in contingency.columns else 0
manual_fail_auto_pass = contingency.loc[False, True] if False in contingency.index and True in contingency.columns else 0

concordant = both_pass + both_fail
discordant = manual_pass_auto_fail + manual_fail_auto_pass

print(f"\nConcordant Pairs (both agree): {concordant:,} ({concordant/len(df)*100:.1f}%)")
print(f"  Both Pass: {both_pass:,}")
print(f"  Both Fail: {both_fail:,}")
print(f"\nDiscordant Pairs (disagree): {discordant:,} ({discordant/len(df)*100:.1f}%)")
print(f"  Manual Pass, Automated Fail: {manual_pass_auto_fail:,}")
print(f"  Manual Fail, Automated Pass: {manual_fail_auto_pass:,}")

### 4.3 Execute McNemar's Test

In [ ]:
# McNemar's test (with continuity correction for large samples)
result = mcnemar(contingency, exact=False, correction=True)

print("\n" + "="*70)
print("MCNEMAR'S TEST: VALIDATION ACCURACY (H₂)")
print("="*70)
print(f"Chi-square statistic: {result.statistic:.4f}")
print(f"p-value: {result.pvalue:.6f}")

# Decision
if result.pvalue < ALPHA:
    print(f"\n✓ REJECT H₀ (p < {ALPHA})")
    print("   Conclusion: Automated validation accuracy SIGNIFICANTLY DIFFERS from manual baseline")
    
    if df['validation_pass_bool'].mean() > MANUAL_VALIDATION_RATE:
        print("   Direction: Automated is MORE ACCURATE than manual")
    else:
        print("   Direction: Automated is LESS ACCURATE than manual")
else:
    print(f"\n❌ FAIL TO REJECT H₀ (p >= {ALPHA})")
    print("   Conclusion: No significant difference in validation accuracy")

# Summary statistics
manual_pass_rate = df['manual_validation_est'].mean() * 100
automated_pass_rate = df['validation_pass_bool'].mean() * 100

print(f"\nValidation Pass Rates:")
print(f"  Manual (Baseline):     {manual_pass_rate:.2f}%")
print(f"  Automated (Measured):  {automated_pass_rate:.2f}%")
print(f"  Absolute Improvement:  {automated_pass_rate - manual_pass_rate:+.2f} percentage points")

# Check success threshold (≥95%)
if automated_pass_rate >= 95:
    print(f"\n✓ Automated validation meets success threshold (≥95%)")
else:
    print(f"\n⚠ Automated validation below target ({automated_pass_rate:.2f}% < 95%)")

### 4.4 Visualization: Confusion Matrix

In [ ]:
# Create confusion matrix visualization
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(df['manual_validation_est'], df['validation_pass_bool'])

fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, 
                              display_labels=['Fail', 'Pass'])
disp.plot(cmap='Blues', ax=ax, values_format='d')
ax.set_title(f'Validation Accuracy: Manual vs Automated\nMcNemar p={result.pvalue:.6f}', 
             fontsize=14, weight='bold')
ax.set_xlabel('Automated (Measured)', fontsize=12)
ax.set_ylabel('Manual (Baseline Estimate)', fontsize=12)

plt.tight_layout()
plt.savefig(output_dir / 'mcnemar_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Saved: mcnemar_confusion_matrix.png")

## 5. Sensitivity Analysis

Test robustness of results across different baseline assumptions.

In [ ]:
# Test across baseline range from literature
# Arndt: 16.8-35 min, Casey: 18-24 min, Adler-Milstein: 20-45 min
baseline_range_min = [15, 20, 25, 30, 45]  # minutes
results = []

for baseline_min in baseline_range_min:
    baseline_sec = baseline_min * 60
    df['manual_latency_temp'] = baseline_sec
    df['latency_diff_temp'] = df['manual_latency_temp'] - df['latency_sec']
    
    # Paired t-test
    t_stat_temp, p_val_temp = stats.ttest_rel(df['manual_latency_temp'], df['latency_sec'])
    p_val_one_tail_temp = p_val_temp / 2 if t_stat_temp > 0 else 1 - (p_val_temp / 2)
    
    # Effect size
    cohens_d_temp = df['latency_diff_temp'].mean() / df['latency_diff_temp'].std()
    
    results.append({
        'baseline_min': baseline_min,
        'baseline_sec': baseline_sec,
        't_statistic': t_stat_temp,
        'p_value': p_val_one_tail_temp,
        'cohens_d': cohens_d_temp,
        'significant': p_val_one_tail_temp < ALPHA,
        'large_effect': cohens_d_temp >= 0.8
    })

results_df = pd.DataFrame(results)

print("\n" + "="*70)
print("SENSITIVITY ANALYSIS: BASELINE ASSUMPTIONS")
print("="*70)
print(results_df.to_string(index=False))

# Robustness assessment
all_significant = results_df['significant'].all()
all_large_effect = results_df['large_effect'].all()

print(f"\nRobustness Assessment:")
print(f"  Significant across all baselines: {'✓ YES' if all_significant else '❌ NO'}")
print(f"  Large effect across all baselines: {'✓ YES' if all_large_effect else '❌ NO'}")

if all_significant and all_large_effect:
    print("\n✓ Results are ROBUST across all baseline assumptions")
else:
    print("\n⚠ Results vary with baseline assumptions - interpret with caution")

In [ ]:
# Visualization: Sensitivity analysis plots
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel A: P-values across baseline
axes[0].plot(results_df['baseline_min'], results_df['p_value'], 
             marker='o', linewidth=2, markersize=8, color='steelblue')
axes[0].axhline(ALPHA, color='red', linestyle='--', linewidth=2, label=f'α = {ALPHA}')
axes[0].set_xlabel('Baseline Manual Time (minutes)', fontsize=12)
axes[0].set_ylabel('p-value (one-tailed)', fontsize=12)
axes[0].set_title('Statistical Significance Across Baseline Assumptions', fontsize=14, weight='bold')
axes[0].legend()
axes[0].set_yscale('log')
axes[0].grid(True, alpha=0.3)

# Panel B: Effect sizes across baseline
axes[1].plot(results_df['baseline_min'], results_df['cohens_d'], 
             marker='o', linewidth=2, markersize=8, color='green')
axes[1].axhline(0.8, color='red', linestyle='--', linewidth=2, label='Large Effect (d=0.8)')
axes[1].axhline(0.5, color='orange', linestyle=':', linewidth=2, label='Medium Effect (d=0.5)')
axes[1].set_xlabel('Baseline Manual Time (minutes)', fontsize=12)
axes[1].set_ylabel('Cohen\'s d', fontsize=12)
axes[1].set_title('Effect Size Across Baseline Assumptions', fontsize=14, weight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'sensitivity_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Saved: sensitivity_analysis.png")

## 6. Subgroup Analysis

Test if improvements hold across different data sources and complexities.

In [ ]:
# Subgroup analysis by data source
subgroup_results = []

for source in df['data_source'].unique():
    subset = df[df['data_source'] == source]
    
    # Skip if too few samples
    if len(subset) < 30:
        continue
    
    # Paired t-test
    t_stat_sub, p_val_sub = stats.ttest_rel(subset['manual_latency_est'], subset['latency_sec'])
    p_val_one_tail_sub = p_val_sub / 2 if t_stat_sub > 0 else 1 - (p_val_sub / 2)
    
    # Effect size
    mean_diff_sub = subset['latency_diff'].mean()
    std_diff_sub = subset['latency_diff'].std()
    cohens_d_sub = mean_diff_sub / std_diff_sub
    
    # CI for Cohen's d (approximation)
    n = len(subset)
    se_d = np.sqrt((n-1)/(n-3)) * np.sqrt((1/n) + (cohens_d_sub**2)/(2*n))
    ci_lower = cohens_d_sub - 1.96 * se_d
    ci_upper = cohens_d_sub + 1.96 * se_d
    
    subgroup_results.append({
        'subgroup': source,
        'n': n,
        'mean_latency': subset['latency_sec'].mean(),
        'cohens_d': cohens_d_sub,
        'ci_lower': ci_lower,
        'ci_upper': ci_upper,
        'p_value': p_val_one_tail_sub,
        'significant': p_val_one_tail_sub < ALPHA
    })

subgroup_df = pd.DataFrame(subgroup_results)

print("\n" + "="*70)
print("SUBGROUP ANALYSIS BY DATA SOURCE")
print("="*70)
print(subgroup_df.to_string(index=False))

# Check consistency
all_subgroups_significant = subgroup_df['significant'].all()
print(f"\nAll subgroups significant: {'✓ YES' if all_subgroups_significant else '⚠ NO'}")

In [ ]:
# Visualization: Forest plot
fig, ax = plt.subplots(figsize=(10, max(6, len(subgroup_df) * 0.8)))

y_pos = np.arange(len(subgroup_df))
ax.errorbar(subgroup_df['cohens_d'], y_pos, 
            xerr=[subgroup_df['cohens_d'] - subgroup_df['ci_lower'],
                  subgroup_df['ci_upper'] - subgroup_df['cohens_d']],
            fmt='o', markersize=10, capsize=5, linewidth=2, color='steelblue')

ax.set_yticks(y_pos)
ax.set_yticklabels([f"{row['subgroup'].title()} (n={row['n']:,})" for _, row in subgroup_df.iterrows()])
ax.set_xlabel('Cohen\'s d (95% CI)', fontsize=12)
ax.set_title('Effect Size by Data Source (Forest Plot)', fontsize=14, weight='bold')
ax.axvline(0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
ax.axvline(0.8, color='red', linestyle=':', linewidth=2, label='Large Effect Threshold (d=0.8)')
ax.legend()
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig(output_dir / 'forest_plot_by_source.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Saved: forest_plot_by_source.png")

## 7. Export Results

In [ ]:
# Compile final results for reporting
final_results = {
    # Sample
    'sample_size': len(df),
    
    # Latency (H1)
    'manual_baseline_sec': MANUAL_BASELINE_SEC,
    'manual_baseline_min': MANUAL_BASELINE_MIN,
    'automated_mean_sec': df['latency_sec'].mean(),
    'automated_median_sec': df['latency_sec'].median(),
    'automated_p50_sec': df['latency_sec'].quantile(0.5),
    'automated_p95_sec': df['latency_sec'].quantile(0.95),
    'mean_improvement_sec': df['latency_diff'].mean(),
    'mean_improvement_min': df['latency_diff'].mean() / 60,
    'speedup_factor': speedup,
    
    # Paired t-test
    'paired_ttest_statistic': t_stat,
    'paired_ttest_pvalue_two_tail': p_value_two_tail,
    'paired_ttest_pvalue_one_tail': p_value_one_tail,
    'paired_ttest_significant': p_value_one_tail < ALPHA,
    
    # Effect size
    'cohens_d': cohens_d,
    'effect_size_interpretation': interpretation,
    
    # Success thresholds
    'meets_p50_threshold_30s': df['latency_sec'].quantile(0.5) <= 30,
    'meets_p95_threshold_120s': df['latency_sec'].quantile(0.95) <= 120,
    
    # Validation (H2)
    'manual_validation_rate': manual_pass_rate / 100,
    'automated_validation_rate': automated_pass_rate / 100,
    'validation_improvement_pp': automated_pass_rate - manual_pass_rate,
    'mcnemar_chi2': result.statistic,
    'mcnemar_pvalue': result.pvalue,
    'mcnemar_significant': result.pvalue < ALPHA,
    'meets_validation_threshold_95pct': automated_pass_rate >= 95,
    
    # Robustness
    'sensitivity_all_significant': all_significant,
    'sensitivity_all_large_effect': all_large_effect,
    'subgroup_all_significant': all_subgroups_significant
}

# Save to CSV
results_table = pd.DataFrame([final_results])
results_path = Path('reports/statistical_results.csv')
results_table.to_csv(results_path, index=False)

print("\n" + "="*70)
print("FINAL RESULTS SUMMARY")
print("="*70)
for key, value in final_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

print(f"\n✓ Results saved to: {results_path}")

## 8. Summary and Conclusions

In [ ]:
print("\n" + "="*70)
print("HYPOTHESIS TESTING SUMMARY")
print("="*70)

print("\n📊 H₁: LATENCY IMPROVEMENT")
print("-" * 70)
if final_results['paired_ttest_significant']:
    print("✓ REJECT H₀: Automated pipeline IS significantly faster (p < 0.05)")
    print(f"  - Mean improvement: {final_results['mean_improvement_min']:.2f} minutes/record")
    print(f"  - Speedup factor: {final_results['speedup_factor']:.1f}x")
    print(f"  - Effect size: {final_results['effect_size_interpretation']} (d={final_results['cohens_d']:.2f})")
else:
    print("❌ FAIL TO REJECT H₀: No significant speed improvement")

print("\n📊 H₂: VALIDATION ACCURACY")
print("-" * 70)
if final_results['mcnemar_significant']:
    print("✓ REJECT H₀: Automated validation IS significantly different (p < 0.05)")
    print(f"  - Automated rate: {final_results['automated_validation_rate']*100:.2f}%")
    print(f"  - Improvement: {final_results['validation_improvement_pp']:+.2f} percentage points")
else:
    print("❌ FAIL TO REJECT H₀: No significant accuracy difference")

print("\n🎯 SUCCESS THRESHOLDS")
print("-" * 70)
print(f"  P50 ≤ 30s:  {'✓ MET' if final_results['meets_p50_threshold_30s'] else '❌ NOT MET'} ({final_results['automated_p50_sec']:.2f}s)")
print(f"  P95 ≤ 120s: {'✓ MET' if final_results['meets_p95_threshold_120s'] else '❌ NOT MET'} ({final_results['automated_p95_sec']:.2f}s)")
print(f"  Validation ≥ 95%: {'✓ MET' if final_results['meets_validation_threshold_95pct'] else '❌ NOT MET'} ({final_results['automated_validation_rate']*100:.2f}%)")

print("\n🔍 ROBUSTNESS")
print("-" * 70)
print(f"  Sensitivity analysis: {'✓ ROBUST' if final_results['sensitivity_all_significant'] else '⚠ VARIES'}")
print(f"  Subgroup consistency: {'✓ CONSISTENT' if final_results['subgroup_all_significant'] else '⚠ INCONSISTENT'}")

print("\n" + "="*70)
print("Next Steps:")
print("  1. Review results in reports/statistical_results.csv")
print("  2. Proceed to 03_visualization_dashboard.ipynb for publication figures")
print("  3. Interpret findings in context of rural healthcare implications")
print("="*70)